# ♟️ Chess-AI: Model Training & Curriculum Fine-Tuning Pipeline

This notebook defines the **Dual-Head ChessResNet** architecture and implements the **Curriculum Training Pipeline**:
- Supervised learning on master move shards (`shard_curriculum_*.npz`)
- Blunder penalization on bad move shards (`bad_*.npz`)
- Differential learning rates for backbone vs. policy/value heads
- Cosine annealing learning rate scheduling


### 1. Imports and Device Setup

In [1]:
import os
import glob
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import chess

# Set device (CUDA GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Training on Device: {device}")
if torch.cuda.is_available():
    print(f"[*] GPU: {torch.cuda.get_device_name(0)}")


### 2. Dual-Head ChessResNet Neural Network Architecture

- **Input**: $12 \times 8 \times 8$ board representation (6 White + 6 Black piece channels).
- **Backbone**: 10 Residual Blocks with 128 hidden channels, Batch Normalization, and ReLU.
- **Policy Head**: 76 move planes $\times$ 64 squares = 4,864 action nodes (sliding rays, knight jumps, underpromotions).
- **Value Head**: $1 \times 8 \times 8 \to 64 \to 1$ with Tanh activation $[-1.0, 1.0]$.


In [2]:
class ResBlock(nn.Module):
    def __init__(self, channels: int = 128):
        super(ResBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        out = F.relu(out)
        return out


class ChessResNet(nn.Module):
    def __init__(self, num_blocks: int = 10, hidden_channels: int = 128):
        super(ChessResNet, self).__init__()
        
        self.start_conv = nn.Conv2d(12, hidden_channels, kernel_size=3, padding=1)
        self.start_bn = nn.BatchNorm2d(hidden_channels)
        self.res_blocks = nn.ModuleList([ResBlock(hidden_channels) for _ in range(num_blocks)])
        
        # --- Policy Head (76 Move Planes x 64 squares = 4,864 action nodes) ---
        self.policy_conv = nn.Conv2d(hidden_channels, 76, kernel_size=1)
        self.policy_bn = nn.BatchNorm2d(76)
        self.policy_fc = nn.Linear(76 * 8 * 8, 4864)
        
        # --- Value Head (Position Evaluation) ---
        self.eval_conv = nn.Conv2d(hidden_channels, 1, kernel_size=1)
        self.eval_bn = nn.BatchNorm2d(1)
        self.eval_fc1 = nn.Linear(8 * 8, 64)
        self.eval_fc2 = nn.Linear(64, 1)

    def forward(self, x: torch.Tensor):
        x = F.relu(self.start_bn(self.start_conv(x)))
        for block in self.res_blocks:
            x = block(x)
            
        # Policy output
        p = F.relu(self.policy_bn(self.policy_conv(x)))
        p = p.view(-1, 4864)
        policy_out = self.policy_fc(p)
        
        # Value output
        v = F.relu(self.eval_bn(self.eval_conv(x)))
        v = v.view(-1, 64)
        v = F.relu(self.eval_fc1(v))
        v = self.eval_fc2(v)
        value_out = torch.tanh(v)
        
        return policy_out, value_out

# Initialize model
model = ChessResNet(num_blocks=10, hidden_channels=128).to(device)
print(f"[*] Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters.")


### 3. Load Pretrained Foundation Weights (Optional)

If you have an existing foundation checkpoint (e.g. `chess_model.pth` or `models/chess_model.pth`), load it here before fine-tuning.

In [3]:
checkpoint_paths = ['models/chess_model.pth', 'chess_model.pth', 'Models/chess_model.pth']
loaded = False

for cp in checkpoint_paths:
    if os.path.exists(cp):
        print(f"[*] Loading pretrained weights from: {cp}")
        state_dict = torch.load(cp, map_location=device, weights_only=True)
        model.load_state_dict(state_dict)
        loaded = True
        break

if not loaded:
    print("[*] No checkpoint found. Initializing training from scratch.")


### 4. Dataset Loader & Training Configuration

The dataset loads `.npz` shards containing `features` (12x8x8 board states), `policy_labels` (target move indices), and `value_labels` (position outcomes). Shards containing `'bad'` in their filename are treated as blunder shards (where policy loss is skipped to avoid learning bad moves, and value loss is used to learn to penalize them).

In [4]:
class ChessDataset(Dataset):
    def __init__(self, npz_path: str):
        data = np.load(npz_path)
        self.features = data['features']
        self.policy_labels = data['policy_labels']
        self.value_labels = data['value_labels']

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx: int):
        board = torch.tensor(self.features[idx], dtype=torch.float32)
        policy = torch.tensor(self.policy_labels[idx], dtype=torch.long)
        value = torch.tensor(self.value_labels[idx], dtype=torch.float32)
        return board, policy, value


def is_bad_shard(shard_path: str) -> bool:
    return "bad" in os.path.basename(shard_path).lower()


# --- CONFIGURATION ---
SHARDS_DIR = "shards"  # Place your training .npz shards in this directory
epoch_limit = 2
batch_size = 128

# Differential learning rates: train base backbone slower than heads
base_params = [p for n, p in model.named_parameters() if "policy" not in n.lower() and "eval" not in n.lower()]
head_params = [p for n, p in model.named_parameters() if "policy" in n.lower() or "eval" in n.lower()]

optimizer = optim.AdamW([
    {'params': base_params, 'lr': 1e-5},
    {'params': head_params, 'lr': 1e-4}
], weight_decay=1e-4)

criterion_cross = nn.CrossEntropyLoss(label_smoothing=0.1, ignore_index=-1)
criterion_mse = nn.MSELoss()


### 5. Curriculum Training Loop

In [5]:
shard_files = glob.glob(os.path.join(SHARDS_DIR, "*.npz"))

if not shard_files:
    print(f"[!] No .npz shard files found in '{SHARDS_DIR}'.")
    print("[!] Place your dataset shards in 'shards/' or update SHARDS_DIR to begin training.")
else:
    total_shards = len(shard_files)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=total_shards * epoch_limit,
        eta_min=1e-6
    )

    print(f"[*] Found {total_shards} shards. Starting curriculum training...")

    for epoch_idx in range(epoch_limit):
        bad_shards = [s for s in shard_files if is_bad_shard(s)]
        good_shards = [s for s in shard_files if not is_bad_shard(s)]

        random.shuffle(bad_shards)
        random.shuffle(good_shards)

        # Alternate between good and bad shards for balanced gradient dynamics
        mixed_shards = []
        while good_shards or bad_shards:
            if good_shards:
                mixed_shards.append(good_shards.pop())
            if bad_shards:
                mixed_shards.append(bad_shards.pop())

        for current_shard_idx, shard_path in enumerate(mixed_shards):
            shard_name = os.path.basename(shard_path)
            bad_shard = is_bad_shard(shard_path)

            print(f"\n--- Epoch {epoch_idx + 1}/{epoch_limit} | Loading {shard_name} ---")

            dataset = ChessDataset(shard_path)
            train_loader = DataLoader(
                dataset,
                batch_size=batch_size,
                shuffle=True,
                pin_memory=torch.cuda.is_available()
            )

            model.train()
            running_loss = 0.0
            correct_policy = 0
            valid_policy_samples = 0
            correct_value = 0
            total_value_samples = 0
            seen_batches = 0

            loop = tqdm(train_loader, desc=f"Shard {current_shard_idx + 1}/{total_shards}")

            for board, policy, value in loop:
                board = board.to(device, non_blocking=True)
                policy = policy.to(device, non_blocking=True)
                value = value.to(device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                output_policy, output_value = model(board)
                output_value = output_value.view_as(value)

                loss_mse = criterion_mse(output_value, value)

                if bad_shard:
                    total_loss = 0.3 * loss_mse
                else:
                    loss_cross = criterion_cross(output_policy, policy)
                    total_loss = loss_cross + (1.5 * loss_mse)

                total_loss.backward()
                optimizer.step()

                seen_batches += 1
                running_loss += total_loss.item()

                # Accuracy metrics
                pred_value = torch.sign(output_value)
                true_value = torch.sign(value)
                correct_value += (pred_value == true_value).sum().item()
                total_value_samples += value.numel()

                if not bad_shard:
                    valid_mask = (policy != -1)
                    if valid_mask.any():
                        pred_moves = output_policy.argmax(dim=1)
                        correct_policy += (pred_moves[valid_mask] == policy[valid_mask]).sum().item()
                        valid_policy_samples += valid_mask.sum().item()

                if bad_shard:
                    loop.set_postfix(
                        loss=f"{running_loss / seen_batches:.4f}",
                        Val_Acc=f"{100. * correct_value / max(total_value_samples, 1):.1f}%",
                        Pol_Acc="SKIPPED"
                    )
                else:
                    loop.set_postfix(
                        loss=f"{running_loss / seen_batches:.4f}",
                        Pol_Acc=f"{100. * correct_policy / max(valid_policy_samples, 1):.1f}%",
                        Val_Acc=f"{100. * correct_value / max(total_value_samples, 1):.1f}%"
                    )

            del dataset, train_loader
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            scheduler.step()
            current_lr = scheduler.get_last_lr()
            print(f"--> Learning rates for next shard: Base={current_lr[0]:.2e}, Heads={current_lr[1]:.2e}")

    os.makedirs("models", exist_ok=True)
    save_path = "models/chess_model_v3.pth"
    torch.save(model.state_dict(), save_path)
    print(f"\n[✔] Training Complete! Checkpoint saved to: {save_path}")
